# Learn-a-World v3: Phase 3 Colab GPU Run 2 (Plan2Explore Curiosity Collection & Incremental Retrain)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sahiltolani30/learn-a-world/blob/main/notebooks/04_phase3_curiosity_colab.ipynb)

**Phase 3 — Tranche T1 Campaign**
- **Policy**: Plan2Explore Curiosity-Guided Recurrent MPC (`lambda_task=0.3, lambda_cur=1.0`).
- **Collection Budget**: +160,000 transitions on reserved seeds (`5000+`).
- **Drive Resilience**: Atomic shard flushes and manifest updates (`dataset_index.json`), state checkpoints every 1,000 steps.
- **Incremental Retraining**: 5,000 steps on ALL data (original 8k shards + new curiosity shards).
- **Screening & Gate Check**: 50 episodes on untouched seeds `2050–2099`. Gate requirement: score > **2.31%** (pre-Phase-3 baseline).

---

### Step 1: Mount Google Drive
Mounts Google Drive to load `checkpoints/jepa_crafter_v3_phase2.pt`, save new curiosity shards, and persist checkpoints.

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/learn-a-world')
(DRIVE_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / 'data' / 'curiosity_shards').mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / 'results').mkdir(parents=True, exist_ok=True)

print('Google Drive workspace initialized at:', DRIVE_DIR)

### Step 2: Clone / Pull Repository on Branch `main` & Install Requirements

In [ ]:
%%bash
cd /content
if [ ! -d "learn-a-world" ]; then
  git clone https://github.com/sahiltolani30/learn-a-world.git
fi
cd learn-a-world
git fetch origin
git checkout main
git pull origin main
echo "--> Latest commit:"
git log -1 --oneline

pip install -q -r requirements.txt
pip install -q crafter gymnasium pytest
nvidia-smi

### Step 3: Run Pre-Flight Automated Gate Tests & Setup Checkpoint
Verifies all 6 Phase-3 curiosity and collector tests pass, and copies base v3 checkpoint locally.

In [ ]:
import os, shutil
from pathlib import Path

# Copy v3 Phase 2 checkpoint from Drive if present
drive_ckpt = Path('/content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_phase2.pt')
local_ckpt = Path('/content/learn-a-world/checkpoints/jepa_crafter_v3_phase2.pt')
local_ckpt.parent.mkdir(parents=True, exist_ok=True)

if drive_ckpt.exists() and not local_ckpt.exists():
    shutil.copy(drive_ckpt, local_ckpt)
    print(f'Copied {drive_ckpt} -> {local_ckpt}')
elif local_ckpt.exists():
    print(f'Local checkpoint ready: {local_ckpt} ({local_ckpt.stat().st_size / 1e6:.1f} MB)')
else:
    print('Notice: Checkpoint not found in Drive. Make sure jepa_crafter_v3_phase2.pt is uploaded to Drive.')

# Run pre-flight tests
!cd /content/learn-a-world && python3 -m pytest tests/test_phase3_curiosity.py -v

### Step 4: Stage 1 — T1 Curiosity Collection (+160,000 Transitions)
Collects self-guided transitions using Plan2Explore 3-predictor ensemble disagreement.
Shards are saved to Drive with atomic manifest updates after each shard. You can stop and resume anytime!

In [ ]:
import sys
sys.path.insert(0, '/content/learn-a-world')
from training.run_v3_colab_run2 import collect_t1

manifest = collect_t1(
    drive_dir='/content/drive/MyDrive/learn-a-world',
    model_checkpoint='/content/learn-a-world/checkpoints/jepa_crafter_v3_phase2.pt',
    target_new_transitions=160_000,
    seed_start=5000,
    device='cuda',
)

print(f"Collection complete! Total transitions: {manifest['total_transitions']:,}")
print(f"Event counts: stone={manifest['event_counts'].get('collect_stone', 0)}, "
      f"coal={manifest['event_counts'].get('collect_coal', 0)}, "
      f"iron={manifest['event_counts'].get('collect_iron', 0)}")

### Step 5: Stage 2 — Incremental Retrain on ALL Data (5,000 steps)
Trains JEPA v3 on the combined dataset (original shards + new curiosity shards).
Saves checkpoints to Drive every 500 steps, keeping the best model by validation loss.

In [ ]:
from training.run_v3_colab_run2 import retrain_t1

t1_ckpt_path = retrain_t1(
    drive_dir='/content/drive/MyDrive/learn-a-world',
    base_checkpoint='/content/learn-a-world/checkpoints/jepa_crafter_v3_phase2.pt',
    num_steps=5_000,
    save_every=500,
    device='cuda',
)

print(f'Retraining complete! Best checkpoint saved at: {t1_ckpt_path}')

### Step 6: Stage 3 — 50-Episode Screening (Untouched Seeds 2050–2099)
Evaluates the retrained model under the identical screening protocol without any heuristic cheats.

In [ ]:
from training.run_v3_colab_run2 import screen_t1

screen_results = screen_t1(
    checkpoint_path=t1_ckpt_path,
    drive_dir='/content/drive/MyDrive/learn-a-world',
    num_episodes=50,
    seed_start=2050,
    device='cuda',
)

print('Screening Complete:')
print(f"  Crafter Score: {screen_results['crafter_score']:.2f}%")
print(f"  Mean Reward:   {screen_results['mean_episode_reward']:.2f}")
print(f"  Survival Steps:{screen_results['mean_survival_length']:.1f}")
print(f"  Unique Achiev: {screen_results['unlocked_achievement_count']}/22")

### Step 7: T1 Gate Verification Check
Evaluates if the T1 model satisfies the gate condition (`score > 2.31%`).

In [ ]:
from training.run_v3_colab_run2 import gate_check_t1

passed = gate_check_t1(screen_results)
if passed:
    print('\n>>> SUCCESS: T1 Gate PASSED! Proceed to download results and review.')
else:
    print('\n>>> NOTICE: T1 Gate NOT met. Analyze event manifests and diagnostic logs before proceeding.')

### Step 8: Commit Screening Results to Git
Copies the screening JSON into the repo results directory and pushes to origin.

In [ ]:
%%bash
cd /content/learn-a-world
cp /content/drive/MyDrive/learn-a-world/results/screen_v3_t1_results.json results/ 2>/dev/null || true
git status